# arXiv Dataset Preprocessing Pipeline

## Overview

This notebook implements a block-based preprocessing pipeline for the arXiv metadata dataset. Its main objective is to clean and normalise the titles and abstracts of scientific articles and store the resulting data in Parquet files.

The dataset is processed in blocks instead of being loaded entirely into memory. This approach makes it possible to work with millions of arXiv records in Google Colab while reducing memory usage and allowing the process to be resumed after an interruption.

The notebook is divided into two main stages:

1. Initial preprocessing of the complete dataset.
2. Validation of existing blocks and resumption of missing or corrupted blocks.

## 1. Mounting Google Drive

The notebook mounts Google Drive in the Colab environment:

```python
from google.colab import drive
drive.mount('/content/drive')
```

This provides access to the original arXiv metadata file and allows the generated Parquet blocks to be stored persistently in Google Drive.

## 2. Installing and importing libraries

The notebook installs and imports the libraries required for the preprocessing pipeline:

* `spaCy` for tokenisation and lemmatisation.
* `pandas` for tabular data processing.
* `PyArrow` for reading and writing Parquet files.
* `tqdm` for progress bars.
* `json` for parsing the arXiv JSONL file.
* `re` for regular-expression-based text cleaning.
* `pathlib` for path management.
* `os` and `time` for multiprocessing configuration and execution-time measurement.

The English spaCy model `en_core_web_sm` is downloaded and loaded because the arXiv titles and abstracts are mainly written in English.

## 3. Defining input and output paths

The notebook defines the following paths:

* The original arXiv metadata snapshot in JSONL format.
* The directory in which the preprocessed Parquet blocks are stored.
* An optional path for a final combined Parquet file.
* A text file used to cache the total number of records in the dataset.

The output directory is automatically created when it does not already exist.

The code also verifies that the input JSONL file exists before starting the preprocessing process. If it cannot be found, a `FileNotFoundError` is raised.

## 4. Counting the dataset records

The arXiv metadata snapshot is stored as a JSONL file, meaning that each line contains the metadata of one scientific article.

The notebook calculates the total number of documents by counting the lines in the file. Because this operation may take several minutes, the result is stored in an auxiliary text file:

```text
num_lineas_arxiv.txt
```

During later executions, the notebook reads the total number of documents directly from this file instead of counting all the lines again.

The total number of expected blocks is then calculated as

$$
\text{number of blocks}
=
\left\lceil
\frac{\text{number of documents}}
{\text{block size}}
\right\rceil.
$$

By default, each block contains up to 50,000 documents.

## 5. Configuring spaCy

The notebook loads the `en_core_web_sm` language model while disabling the parser and named-entity recognition components:

```python
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)
```

These components are not required for the preprocessing task, so disabling them reduces execution time and memory consumption. The lemmatiser remains active.

The notebook also obtains the English stop-word list provided by spaCy and creates a filtered version that can preserve selected terms.

The number of spaCy processes is automatically selected according to the available CPU resources:

```python
n_process_spacy = max(1, min(4, os.cpu_count()))
```

A maximum of four processes is used to avoid excessive memory consumption in Google Colab. Although a GPU may be available, tokenisation and lemmatisation in this pipeline are mainly CPU-based operations.

## 6. Initial text cleaning

The function `limpieza_inicial` applies a preliminary cleaning procedure before the text is processed with spaCy.

The function performs the following operations:

1. Converts the input into a string.
2. Decodes HTML entities such as `&amp;`.
3. Converts the text to lowercase.
4. Removes HTML tags.
5. Removes URLs.
6. Removes user mentions.
7. Removes mathematical expressions enclosed between dollar signs.
8. Removes common LaTeX commands such as `\alpha` or `\frac`.
9. Removes frequent LaTeX symbols such as braces, underscores and carets.
10. Replaces repeated whitespace with a single space.

This cleaning stage reduces noise commonly found in scientific titles and abstracts, particularly HTML and LaTeX markup.

## 7. Token-level preprocessing

The function `preprocesar_documento` receives a document already tokenised by spaCy and produces its final cleaned textual representation.

For each token, the function:

* Removes whitespace tokens.
* Removes punctuation marks.
* Removes numerical tokens.
* Removes non-alphabetic tokens.
* Removes stop words.
* Converts the token to its lemma.
* Removes empty or invalid lemmas.
* Applies stop-word filtering again after lemmatisation.

The retained lemmas are joined into a single space-separated string.

For example, a sentence such as

```text
The proposed algorithms were evaluated on several datasets.
```

could be transformed into a representation similar to

```text
propose algorithm evaluate dataset
```

The exact result depends on spaCy's tokenisation, stop-word list and lemmatisation rules.

## 8. Reading the JSONL dataset in blocks

The function `leer_bloques_arxiv_jsonl` reads the arXiv JSONL file sequentially instead of loading it entirely into memory.

For each article, the following fields are retained:

* `id`
* `title`
* `abstract`
* `categories`
* `authors`
* `update_date`

Malformed JSON lines are skipped.

Once the number of accumulated records reaches the configured block size, the records are converted into a pandas DataFrame and returned using `yield`. The final block may contain fewer records than the other blocks.

This generator-based design considerably reduces memory usage.

## 9. Constructing the textual field

For every block, the title and abstract are combined into a single column named `texto`:

```python
df_bloque["texto"] = (
    df_bloque["title"].fillna("").astype(str)
    + " "
    + df_bloque["abstract"].fillna("").astype(str)
)
```

This combined field contains the textual information that is subsequently cleaned and normalised.

Missing titles or abstracts are replaced with empty strings before concatenation.

## 10. Processing the documents with spaCy

The initially cleaned texts are passed to `nlp.pipe`, which processes multiple documents efficiently in batches:

```python
nlp.pipe(
    textos_limpios_iniciales,
    batch_size=1000,
    n_process=n_process_spacy
)
```

The notebook uses a spaCy batch size of 1,000 documents and displays a progress bar for each block.

Each processed spaCy document is passed to `preprocesar_documento`, and the result is stored in a new column named:

```text
texto_preprocesado
```

This column contains the final lemmatised and filtered document representation.

## 11. Removing empty documents

After preprocessing, some documents may become empty because all their tokens were removed.

The notebook removes these records using the following condition:

```python
df_bloque["texto_preprocesado"].str.strip() != ""
```

This ensures that every retained document contains at least one valid preprocessed token.

## 12. Saving the processed blocks

Each processed block is stored as an independent Parquet file using a sequential filename:

```text
arxiv_bloque_00000.parquet
arxiv_bloque_00001.parquet
arxiv_bloque_00002.parquet
...
```

The retained columns are:

* Article identifier.
* Original title.
* Original abstract.
* Subject categories.
* Authors.
* Update date.
* Combined original text.
* Preprocessed text.

Parquet is used because it provides compact storage, preserves the table schema and can be read efficiently by pandas.

Before processing a block, the notebook checks whether its output file already exists. Existing blocks are skipped, allowing the pipeline to resume after a Colab disconnection without repeating all previous work.

## 13. Validating existing blocks

The second part of the notebook checks the Parquet blocks already stored in Google Drive.

For every expected block, it verifies:

1. Whether the corresponding file exists.
2. Whether the Parquet file can be read.
3. Whether it contains all the expected columns.

The blocks are classified into three groups:

* Valid blocks.
* Corrupted blocks.
* Missing blocks.

If a Parquet file cannot be read or does not contain the required columns, it is considered corrupted. The file is deleted and its block number is added to the list of blocks that must be processed again.

## 14. Resuming missing or corrupted blocks

After validation, the notebook resumes preprocessing only for the blocks identified as missing or corrupted.

Already validated blocks are skipped. This avoids repeating completed work and significantly reduces the time required to recover from an interrupted execution.

For each pending block, the notebook repeats the same procedure:

1. Combine the title and abstract.
2. Apply the initial cleaning function.
3. Process the texts with spaCy.
4. Remove empty documents.
5. Select the relevant columns.
6. Save the result.

## 15. Safe file writing

During the recovery stage, each block is initially written to a temporary file:

```text
arxiv_bloque_00000.tmp.parquet
```

Only after the writing operation has completed successfully is the temporary file renamed to its final name:

```text
arxiv_bloque_00000.parquet
```

This procedure prevents an incomplete or corrupted file from being stored under the final filename if Google Colab disconnects during the write operation.

## 16. Execution-time monitoring

The notebook records the starting and ending times of the preprocessing process and reports the total execution time in:

* Seconds.
* Minutes.
* Hours.

Progress bars are also displayed both for the complete set of blocks and for spaCy processing within each block.

## Final output

The main result of the notebook is a directory containing validated Parquet blocks with the original and preprocessed arXiv data.

Each valid record includes the original metadata together with:

```text
texto
```

which combines the title and abstract, and

```text
texto_preprocesado
```

which contains the cleaned and lemmatised version used in later text-representation experiments.

These files can subsequently be used to construct representations such as TF-IDF, LSA, LDA or SG-NS embeddings without having to repeat the expensive linguistic preprocessing stage.

## Important implementation notes

The variable used to specify retained negations is currently defined as:

```python
negaciones = {"hello"}
```

However, `hello` is not a negation. This appears to be a placeholder or configuration error. Depending on the preprocessing objective, this set could instead contain terms such as:

```python
negaciones = {"no", "not", "never"}
```

The path `archivo_salida_final` is defined for a combined Parquet file, but the current notebook does not merge the individual blocks into that final file.

The validation stage uses `pd.read_parquet` to load each complete block. Therefore, it validates more than only the file metadata and may require considerable memory for large blocks.


In [1]:
# ==============================================================================
# BLOCK-BASED PREPROCESSING OF THE arXiv DATASET
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==============================================================================
# 1. INSTALLING AND IMPORTING LIBRARIES
# ==============================================================================

!pip install spacy tqdm pyarrow -q
!python -m spacy download en_core_web_sm -q

import os
import re
import html
import json
import math
import time
import pandas as pd
import spacy

from pathlib import Path
from tqdm.auto import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# ==============================================================================
# 2. PATH DEFINITIONS
# ==============================================================================

BASE_DIR = Path("/content/drive/MyDrive/TFM/Dataset")

ARXIV_DIR = BASE_DIR / "arxiv"
archivo_entrada = ARXIV_DIR / "arxiv-metadata-oai-snapshot.json"

SALIDA_BLOQUES_DIR = BASE_DIR / "arxiv_preprocesado_bloques"
SALIDA_BLOQUES_DIR.mkdir(parents=True, exist_ok=True)

archivo_salida_final = BASE_DIR / "arxiv_preprocesado.parquet"

print("Archivo de entrada:", archivo_entrada)
print("Carpeta de salida por bloques:", SALIDA_BLOQUES_DIR)
print("Archivo final opcional:", archivo_salida_final)

if not archivo_entrada.exists():
    raise FileNotFoundError(f"No se ha encontrado el archivo: {archivo_entrada}")

Archivo de entrada: /content/drive/MyDrive/TFM/Dataset/arxiv/arxiv-metadata-oai-snapshot.json
Carpeta de salida por bloques: /content/drive/MyDrive/TFM/Dataset/arxiv_preprocesado_bloques
Archivo final opcional: /content/drive/MyDrive/TFM/Dataset/arxiv_preprocesado.parquet


In [4]:
# ==============================================================================
# 3. COUNTING RECORDS IN THE JSONL FILE
# ==============================================================================

archivo_total_lineas = ARXIV_DIR / "num_lineas_arxiv.txt"

if archivo_total_lineas.exists():
    total_documentos = int(archivo_total_lineas.read_text().strip())
    print("Número total de registros leído desde archivo auxiliar:", total_documentos)
else:
    print("Contando líneas del archivo arXiv. Este paso puede tardar unos minutos...")

    total_documentos = 0

    with open(archivo_entrada, "r", encoding="utf-8") as f:
        for _ in tqdm(f, desc="Contando registros"):
            total_documentos += 1

    archivo_total_lineas.write_text(str(total_documentos))

    print("Número total de registros:", total_documentos)

Número total de registros leído desde archivo auxiliar: 3058383


In [5]:
# ==============================================================================
# 4. SPACY CONFIGURATION
# ==============================================================================

# Disable the parser and NER components to speed up processing.
# Keep the lemmatiser enabled.
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

# spaCy stop words
stopwords_spacy = set(nlp.Defaults.stop_words)

# Negations that will be explicitly retained
negaciones = {"hello"}

# Remove retained negations from the stop-word set
stopwords_filtradas = stopwords_spacy - negaciones

print("Número de stopwords originales:", len(stopwords_spacy))
print("Número de stopwords tras conservar negaciones:", len(stopwords_filtradas))

# Number of processes used by spaCy.
# In Colab with a T4 GPU, the GPU does not normally accelerate this preprocessing;
# spaCy mainly uses the CPU for tokenisation and lemmatisation.
n_process_spacy = max(1, min(4, os.cpu_count()))

print("Número de CPUs disponibles:", os.cpu_count())
print("Número de procesos usados por spaCy:", n_process_spacy)

Número de stopwords originales: 326
Número de stopwords tras conservar negaciones: 326
Número de CPUs disponibles: 2
Número de procesos usados por spaCy: 2


In [6]:
# ==============================================================================
# 5. AUXILIARY CLEANING FUNCTIONS
# ==============================================================================

def limpieza_inicial(texto):
    """
    Applies preliminary cleaning before tokenisation with spaCy.

    Operations:
    - Conversion to lowercase.
    - Decoding of HTML entities.
    - Removal of HTML tags.
    - Removal of URLs.
    - Removal of user mentions.
    - Whitespace normalisation.

    For arXiv, some basic LaTeX commands are also removed to reduce noise
    in scientific titles and abstracts.
    """

    texto = str(texto)

    # Decode HTML entities, for example, &amp;
    texto = html.unescape(texto)

    # Convert to lowercase
    texto = texto.lower()

    # Remove HTML tags
    texto = re.sub(r"<.*?>", " ", texto)

    # Remove URLs
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)

    # Remove user mentions
    texto = re.sub(r"@\w+", " ", texto)

    # Basic cleaning of common LaTeX expressions in scientific abstracts
    texto = re.sub(r"\$.*?\$", " ", texto)          # Expressions enclosed in $
    texto = re.sub(r"\\[a-zA-Z]+", " ", texto)      # Commands such as \alpha or \frac
    texto = re.sub(r"[{}_^]", " ", texto)           # Common LaTeX symbols

    # Normalise whitespace
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [7]:
# ==============================================================================
# 6. PREPROCESSING FUNCTION WITH SPACY
# ==============================================================================

def preprocesar_documento(doc):
    """
    Applies tokenisation, removal of numerical tokens,
    removal of stop words except negations, and lemmatisation.
    """

    tokens_limpios = []

    for token in doc:
        token_lower = token.text.lower().strip()

        # Remove whitespace tokens
        if token.is_space:
            continue

        # Remove punctuation, except when it is a negation such as n't
        if token.is_punct and token_lower not in negaciones:
            continue

        # Remove numerical tokens
        if token.like_num or token.is_digit:
            continue

        # Remove non-alphabetic tokens, except negations
        if not token.is_alpha and token_lower not in negaciones:
            continue

        # Remove stop words, except negations
        if token_lower in stopwords_filtradas:
            continue

        # Lemmatise the token
        lema = token.lemma_.lower().strip()

        # Additional check for empty lemmas
        if lema == "" or lema == "-pron-":
            continue

        # Remove stop words again after lemmatisation, except negations
        if lema in stopwords_filtradas:
            continue

        tokens_limpios.append(lema)

    return " ".join(tokens_limpios)

In [8]:
# ==============================================================================
# 7. BLOCK GENERATOR FOR THE JSONL FILE
# ==============================================================================

def leer_bloques_arxiv_jsonl(archivo_jsonl, tam_bloque=50000):
    """
    Reads the arXiv JSONL file in blocks.

    Each line in the file corresponds to one article.
    Relevant columns are retained, and the text column is created later.
    """

    registros = []
    num_bloque = 0

    with open(archivo_jsonl, "r", encoding="utf-8") as f:

        for linea in f:
            try:
                obj = json.loads(linea)
            except json.JSONDecodeError:
                continue

            registros.append({
                "id": obj.get("id", ""),
                "title": obj.get("title", ""),
                "abstract": obj.get("abstract", ""),
                "categories": obj.get("categories", ""),
                "authors": obj.get("authors", ""),
                "update_date": obj.get("update_date", "")
            })

            if len(registros) == tam_bloque:
                yield num_bloque, pd.DataFrame(registros)
                num_bloque += 1
                registros = []

    if len(registros) > 0:
        yield num_bloque, pd.DataFrame(registros)

In [9]:

# ==============================================================================
# 8. APPLYING BLOCK-BASED PREPROCESSING AND SAVING TO PARQUET
# ==============================================================================

inicio = time.time()

batch_size_dataset = 50000
batch_size_spacy = 1000

total_bloques = math.ceil(total_documentos / batch_size_dataset)

print("Número total de documentos:", total_documentos)
print("Tamaño de bloque:", batch_size_dataset)
print("Número total aproximado de bloques:", total_bloques)

barra_bloques = tqdm(
    leer_bloques_arxiv_jsonl(archivo_entrada, tam_bloque=batch_size_dataset),
    total=total_bloques,
    desc="Procesando bloques arXiv"
)

for num_bloque, df_bloque in barra_bloques:

    archivo_bloque = SALIDA_BLOQUES_DIR / f"arxiv_bloque_{num_bloque:05d}.parquet"

    # Skip the block if it already exists.
    # This makes it possible to resume processing if Colab disconnects.
    if archivo_bloque.exists():
        barra_bloques.set_postfix({
            "bloque": num_bloque,
            "estado": "ya existe"
        })
        continue

    # --------------------------------------------------------------------------
    # 1. Create the text column by combining the title and abstract
    # --------------------------------------------------------------------------

    df_bloque["texto"] = (
        df_bloque["title"].fillna("").astype(str)
        + " "
        + df_bloque["abstract"].fillna("").astype(str)
    )

    # --------------------------------------------------------------------------
    # 2. Apply preliminary cleaning before spaCy processing
    # --------------------------------------------------------------------------

    textos_limpios_iniciales = (
        df_bloque["texto"]
        .apply(limpieza_inicial)
        .tolist()
    )

    textos_preprocesados = []

    # --------------------------------------------------------------------------
    # 3. Process the texts with spaCy
    # --------------------------------------------------------------------------

    barra_spacy = tqdm(
        nlp.pipe(
            textos_limpios_iniciales,
            batch_size=batch_size_spacy,
            n_process=n_process_spacy
        ),
        total=len(textos_limpios_iniciales),
        desc=f"spaCy bloque {num_bloque}",
        leave=False
    )

    for i, doc in enumerate(barra_spacy, start=1):
        textos_preprocesados.append(preprocesar_documento(doc))

        porcentaje_bloque = (i / len(textos_limpios_iniciales)) * 100

        barra_spacy.set_postfix({
            "bloque": f"{porcentaje_bloque:.2f}%",
            "faltan_bloque": len(textos_limpios_iniciales) - i
        })

    df_bloque["texto_preprocesado"] = textos_preprocesados

    # --------------------------------------------------------------------------
    # 4. Remove documents that are empty after cleaning
    # --------------------------------------------------------------------------

    df_bloque["texto_preprocesado"] = (
        df_bloque["texto_preprocesado"]
        .fillna("")
        .astype(str)
    )

    df_bloque = df_bloque[
        df_bloque["texto_preprocesado"].str.strip() != ""
    ].copy()

    # --------------------------------------------------------------------------
    # 5. Select the relevant columns
    # --------------------------------------------------------------------------

    df_bloque = df_bloque[[
        "id",
        "title",
        "abstract",
        "categories",
        "authors",
        "update_date",
        "texto",
        "texto_preprocesado"
    ]]

    # --------------------------------------------------------------------------
    # 6. Save the processed block
    # --------------------------------------------------------------------------

    df_bloque.to_parquet(archivo_bloque, index=False)

    barra_bloques.set_postfix({
        "bloque": num_bloque,
        "guardado": archivo_bloque.name
    })

fin = time.time()

print("\nPreprocesamiento por bloques finalizado.")
print(f"Tiempo total: {fin - inicio:.2f} segundos")
print(f"Tiempo total: {(fin - inicio) / 60:.2f} minutos")
print(f"Tiempo total: {(fin - inicio) / 3600:.2f} horas")

In [10]:
# ==============================================================================
# BLOCK-BASED PREPROCESSING OF THE arXiv DATASET
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:

# ==============================================================================
# CHECK PREPROCESSED arXiv BLOCKS
# ==============================================================================

from pathlib import Path
import pandas as pd
import math

BASE_DIR = Path("/content/drive/MyDrive/TFM/Dataset")

ARXIV_DIR = BASE_DIR / "arxiv"
archivo_entrada = ARXIV_DIR / "arxiv-metadata-oai-snapshot.json"

SALIDA_BLOQUES_DIR = BASE_DIR / "arxiv_preprocesado_bloques"

batch_size_dataset = 50000

# If total_documentos has already been defined, this block can be omitted.
archivo_total_lineas = ARXIV_DIR / "num_lineas_arxiv.txt"

if archivo_total_lineas.exists():
    total_documentos = int(archivo_total_lineas.read_text().strip())
else:
    total_documentos = 0
    with open(archivo_entrada, "r", encoding="utf-8") as f:
        for _ in f:
            total_documentos += 1
    archivo_total_lineas.write_text(str(total_documentos))

total_bloques = math.ceil(total_documentos / batch_size_dataset)

print("Total documentos:", total_documentos)
print("Total bloques esperados:", total_bloques)

archivos_bloques = sorted(SALIDA_BLOQUES_DIR.glob("arxiv_bloque_*.parquet"))

print("Bloques encontrados:", len(archivos_bloques))
print("Primeros bloques encontrados:")
print([p.name for p in archivos_bloques[:10]])


Total documentos: 3058383
Total bloques esperados: 62
Bloques encontrados: 48
Primeros bloques encontrados:
['arxiv_bloque_00000.parquet', 'arxiv_bloque_00001.parquet', 'arxiv_bloque_00002.parquet', 'arxiv_bloque_00003.parquet', 'arxiv_bloque_00004.parquet', 'arxiv_bloque_00005.parquet', 'arxiv_bloque_00006.parquet', 'arxiv_bloque_00007.parquet', 'arxiv_bloque_00008.parquet', 'arxiv_bloque_00009.parquet']


In [12]:
# ==============================================================================
# VALIDATE EXISTING BLOCKS AND DETECT MISSING BLOCKS
# ==============================================================================

bloques_validos = []
bloques_corruptos = []
bloques_faltantes = []

for num_bloque in range(total_bloques):

    archivo_bloque = SALIDA_BLOQUES_DIR / f"arxiv_bloque_{num_bloque:05d}.parquet"

    if not archivo_bloque.exists():
        bloques_faltantes.append(num_bloque)
        continue

    try:
        # Attempt to read the metadata or the complete file for validation
        df_tmp = pd.read_parquet(archivo_bloque)

        # Minimum validation of the expected columns
        columnas_esperadas = {
            "id",
            "title",
            "abstract",
            "categories",
            "authors",
            "update_date",
            "texto",
            "texto_preprocesado"
        }

        if not columnas_esperadas.issubset(set(df_tmp.columns)):
            bloques_corruptos.append(num_bloque)
            archivo_bloque.unlink()
            bloques_faltantes.append(num_bloque)
        else:
            bloques_validos.append(num_bloque)

        del df_tmp

    except Exception as e:
        print(f"Bloque corrupto detectado: {num_bloque} -> {e}")
        bloques_corruptos.append(num_bloque)

        # Delete the corrupted file so that it can be reprocessed
        archivo_bloque.unlink()

        bloques_faltantes.append(num_bloque)

print("Bloques válidos:", len(bloques_validos))
print("Bloques corruptos eliminados:", bloques_corruptos)
print("Bloques faltantes:", len(bloques_faltantes))

if len(bloques_faltantes) > 0:
    print("Primeros bloques faltantes:", bloques_faltantes[:20])
else:
    print("Todos los bloques están procesados correctamente.")


Bloques válidos: 48
Bloques corruptos eliminados: []
Bloques faltantes: 14
Primeros bloques faltantes: [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]


In [13]:

# ==============================================================================
# RESUME PREPROCESSING ONLY FOR MISSING BLOCKS
# ==============================================================================

import time
from tqdm.auto import tqdm

inicio = time.time()

batch_size_dataset = 50000
batch_size_spacy = 1000

bloques_faltantes_set = set(bloques_faltantes)

print("Número de bloques pendientes:", len(bloques_faltantes_set))

if len(bloques_faltantes_set) == 0:
    print("No hay bloques pendientes. El preprocesamiento ya está completo.")

else:
    barra_bloques = tqdm(
        leer_bloques_arxiv_jsonl(
            archivo_entrada,
            tam_bloque=batch_size_dataset
        ),
        total=total_bloques,
        desc="Reanudando bloques arXiv"
    )

    for num_bloque, df_bloque in barra_bloques:

        # Skip blocks that have already been processed and validated
        if num_bloque not in bloques_faltantes_set:
            barra_bloques.set_postfix({
                "bloque": num_bloque,
                "estado": "ya procesado"
            })
            continue

        archivo_bloque = (
            SALIDA_BLOQUES_DIR
            / f"arxiv_bloque_{num_bloque:05d}.parquet"
        )
        archivo_temporal = (
            SALIDA_BLOQUES_DIR
            / f"arxiv_bloque_{num_bloque:05d}.tmp.parquet"
        )

        print("\n" + "=" * 80)
        print(f"Procesando bloque pendiente: {num_bloque}")
        print("=" * 80)

        # ----------------------------------------------------------------------
        # 1. Create the text column by combining the title and abstract
        # ----------------------------------------------------------------------

        df_bloque["texto"] = (
            df_bloque["title"].fillna("").astype(str)
            + " "
            + df_bloque["abstract"].fillna("").astype(str)
        )

        # ----------------------------------------------------------------------
        # 2. Apply preliminary cleaning before spaCy processing
        # ----------------------------------------------------------------------

        textos_limpios_iniciales = (
            df_bloque["texto"]
            .apply(limpieza_inicial)
            .tolist()
        )

        textos_preprocesados = []

        # ----------------------------------------------------------------------
        # 3. Process the texts with spaCy
        # ----------------------------------------------------------------------

        barra_spacy = tqdm(
            nlp.pipe(
                textos_limpios_iniciales,
                batch_size=batch_size_spacy,
                n_process=n_process_spacy
            ),
            total=len(textos_limpios_iniciales),
            desc=f"spaCy bloque {num_bloque}",
            leave=False
        )

        for i, doc in enumerate(barra_spacy, start=1):

            textos_preprocesados.append(
                preprocesar_documento(doc)
            )

            porcentaje_bloque = (
                i / len(textos_limpios_iniciales)
            ) * 100

            barra_spacy.set_postfix({
                "bloque": f"{porcentaje_bloque:.2f}%",
                "faltan_bloque": (
                    len(textos_limpios_iniciales) - i
                )
            })

        df_bloque["texto_preprocesado"] = textos_preprocesados

        # ----------------------------------------------------------------------
        # 4. Remove documents that are empty after cleaning
        # ----------------------------------------------------------------------

        df_bloque["texto_preprocesado"] = (
            df_bloque["texto_preprocesado"]
            .fillna("")
            .astype(str)
        )

        df_bloque = df_bloque[
            df_bloque["texto_preprocesado"].str.strip() != ""
        ].copy()

        # ----------------------------------------------------------------------
        # 5. Select the relevant columns
        # ----------------------------------------------------------------------

        df_bloque = df_bloque[[
            "id",
            "title",
            "abstract",
            "categories",
            "authors",
            "update_date",
            "texto",
            "texto_preprocesado"
        ]]

        # ----------------------------------------------------------------------
        # 6. Save the block safely
        # ----------------------------------------------------------------------
        # First, save the block to a temporary file.
        # Once the operation completes successfully, rename it to the final file.
        # This prevents a corrupted .parquet file from being left behind if
        # Colab disconnects.

        df_bloque.to_parquet(
            archivo_temporal,
            index=False
        )

        archivo_temporal.rename(archivo_bloque)

        barra_bloques.set_postfix({
            "bloque": num_bloque,
            "guardado": archivo_bloque.name
        })

fin = time.time()

print("\nReanudación finalizada.")
print(f"Tiempo total: {fin - inicio:.2f} segundos")
print(f"Tiempo total: {(fin - inicio) / 60:.2f} minutos")
print(f"Tiempo total: {(fin - inicio) / 3600:.2f} horas")

Número de bloques pendientes: 14


Reanudando bloques arXiv:   0%|          | 0/62 [00:00<?, ?it/s]


Procesando bloque pendiente: 48


spaCy bloque 48:   0%|          | 0/50000 [00:00<?, ?it/s]


Procesando bloque pendiente: 49


...


Procesando bloque pendiente: 60


spaCy bloque 60:   0%|          | 0/50000 [00:00<?, ?it/s]


Procesando bloque pendiente: 61


spaCy bloque 61:   0%|          | 0/8383 [00:00<?, ?it/s]


Reanudación finalizada.
Tiempo total: 12542.48 segundos
Tiempo total: 209.04 minutos
Tiempo total: 3.48 horas
